# Anomaly Detection: Finding the Odd One Out

Anomaly detection (also called outlier detection) is the identification of rare items, events, or observations that differ significantly from the majority of the data. These anomalies often indicate critical information — fraud, network intrusions, equipment failure, or novel phenomena.

> "One person's noise is another person's signal." — Anomaly detection finds the signal in the noise.

In this notebook we will cover:
- What anomalies are and their types
- Statistical methods: Z-score, IQR, Modified Z-score
- Machine learning approaches: Isolation Forest, One-Class SVM, Elliptic Envelope, LOF
- DBSCAN for outlier detection
- Side-by-side comparison on synthetic data
- Evaluation metrics and the contamination parameter
- Novelty detection vs outlier detection
- Real-world style examples

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap
from sklearn.datasets import make_blobs, make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, precision_recall_fscore_support
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.covariance import EllipticEnvelope
from sklearn.neighbors import LocalOutlierFactor
from sklearn.cluster import DBSCAN
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
rng = np.random.RandomState(42)

print("Libraries loaded successfully.")

---
## 1. What Are Anomalies?

An **anomaly** is a data point that differs significantly from other observations. Hawkins (1980) defines it as:

> "An observation which deviates so much from other observations as to arouse suspicion that it was generated by a different mechanism."

### Three Types of Anomalies

| Type | Description | Example |
|------|-------------|---------|
| **Point Anomaly** | A single instance that is anomalous relative to the rest | A credit card transaction of $10,000 when the typical is $50 |
| **Contextual Anomaly** | An instance anomalous in a specific context (time, location) | 30°C in summer is normal, but 30°C in winter is anomalous |
| **Collective Anomaly** | A set of instances together form an anomaly, though each alone may be normal | A sequence of small fraudulent transactions that alone look normal but together indicate fraud |

---
## 2. Applications

- **Fraud Detection** — Identify unusual credit card transactions, insurance claims, or tax returns.
- **Network Intrusion Detection** — Spot unusual traffic patterns that indicate cyberattacks.
- **Manufacturing / Quality Control** — Detect defective products on an assembly line.
- **Healthcare** — Flag abnormal patient vitals or medical images.
- **IoT / Sensor Monitoring** — Identify sensor failures or anomalous readings.
- **Data Cleaning** — Find and remove erroneous entries before modeling.

---
## 3. Statistical Methods for Anomaly Detection

Statistical methods assume the data follows some known distribution and flag points that fall in low-probability regions.

In [ ]:
# Generate normally distributed data with some outliers
np.random.seed(42)
data = np.random.normal(0, 1, 200)
outliers = np.array([-3.5, 3.8, 4.2, -4.0, 3.0, -3.2, 3.5])  # known outliers
data_with_outliers = np.concatenate([data, outliers])

df_stats = pd.DataFrame({'value': data_with_outliers})
print(f"Dataset size: {len(df_stats)}")
print(f"Mean: {df_stats['value'].mean():.3f}, Std: {df_stats['value'].std():.3f}")
print(f"Skewness: {df_stats['value'].skew():.3f}, Kurtosis: {df_stats['value'].kurtosis():.3f}")

### 3.1 Z-Score Method

$$Z_i = \frac{x_i - \mu}{\sigma}$$

A common threshold is $|Z| > 3$, which flags points beyond ~3 standard deviations from the mean.

In [ ]:
df_stats['z_score'] = np.abs(stats.zscore(df_stats['value']))
z_threshold = 3
df_stats['z_outlier'] = df_stats['z_score'] > z_threshold

print(f"Z-score threshold: {z_threshold}")
print(f"Outliers detected by Z-score: {df_stats['z_outlier'].sum()}")
print(f"Outlier indices: {df_stats[df_stats['z_outlier']].index.tolist()}")

### 3.2 IQR Method

$$\text{Outlier} < Q_1 - 1.5 \times IQR \quad \text{or} \quad \text{Outlier} > Q_3 + 1.5 \times IQR$$

In [ ]:
Q1 = df_stats['value'].quantile(0.25)
Q3 = df_stats['value'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df_stats['iqr_outlier'] = (df_stats['value'] < lower_bound) | (df_stats['value'] > upper_bound)

print(f"Q1 = {Q1:.3f}, Q3 = {Q3:.3f}, IQR = {IQR:.3f}")
print(f"Bounds: [{lower_bound:.3f}, {upper_bound:.3f}]")
print(f"Outliers detected by IQR: {df_stats['iqr_outlier'].sum()}")
print(f"Outlier indices: {df_stats[df_stats['iqr_outlier']].index.tolist()}")

### 3.3 Modified Z-Score (MAD)

Uses the median and Median Absolute Deviation (MAD) — more robust to outliers in the training set.

$$MAD = \text{median}(|x_i - \tilde{x}|)$$
$$M_i = \frac{0.6745 \times (x_i - \tilde{x})}{MAD}$$

Common threshold: $|M_i| > 3.5$.

In [ ]:
median = np.median(df_stats['value'])
mad = np.median(np.abs(df_stats['value'] - median))
modified_z = 0.6745 * (df_stats['value'] - median) / mad
df_stats['mad_outlier'] = np.abs(modified_z) > 3.5

print(f"Median = {median:.3f}, MAD = {mad:.3f}")
print(f"Outliers detected by Modified Z-score: {df_stats['mad_outlier'].sum()}")
print(f"Outlier indices: {df_stats[df_stats['mad_outlier']].index.tolist()}")

### Z-Score / IQR Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Distribution with Z-score thresholds
ax = axes[0]
ax.hist(df_stats['value'], bins=30, alpha=0.7, color='steelblue', edgecolor='black')
for t, c, ls in [(z_threshold, 'red', '--'), (-z_threshold, 'red', '--')]:
    ax.axvline(df_stats['value'].mean() + t * df_stats['value'].std(), color=c, ls=ls, lw=2, label=f'Z = \u00b1{t}')
ax.axvline(df_stats['value'].mean(), color='green', ls='-', lw=2, label='Mean')
scatter_x = df_stats['value'][df_stats['z_outlier']]
ax.scatter(scatter_x, np.zeros(len(scatter_x)) + 1, color='red', s=80, zorder=5, label=f'{len(scatter_x)} outliers')
ax.set_title('Z-Score Method (|Z| > 3)')
ax.set_xlabel('Value')
ax.set_ylabel('Frequency')
ax.legend()

# 2. Box plot with IQR
ax = axes[1]
bp = ax.boxplot(df_stats['value'], patch_artist=True, widths=0.3)
bp['boxes'][0].set_facecolor('lightblue')
ax.axhline(lower_bound, color='red', ls='--', lw=2, label=f'Lower: {lower_bound:.2f}')
ax.axhline(upper_bound, color='red', ls='--', lw=2, label=f'Upper: {upper_bound:.2f}')
outlier_vals = df_stats['value'][df_stats['iqr_outlier']]
ax.scatter(np.ones(len(outlier_vals)), outlier_vals, color='red', s=80, zorder=5, label=f'{len(outlier_vals)} outliers')
ax.set_title('IQR Method (1.5 × IQR)')
ax.set_xticks([])
ax.legend()

# 3. Scatter plot comparing all methods
ax = axes[2]
x = np.arange(len(df_stats))
ax.scatter(x, df_stats['value'], c='steelblue', s=20, alpha=0.6, label='Normal')
for method, marker, color in [('z_outlier', 'o', 'red'), ('iqr_outlier', 's', 'orange'), ('mad_outlier', '^', 'purple')]:
    idx = df_stats[df_stats[method]].index
    ax.scatter(idx, df_stats.loc[idx, 'value'], marker=marker, color=color, s=60,
               label=method.replace('_outlier','').upper(), zorder=5, alpha=0.8, edgecolors='black')
ax.set_title('Comparison of Statistical Methods')
ax.set_xlabel('Index')
ax.set_ylabel('Value')
ax.legend()

plt.suptitle('Statistical Anomaly Detection Methods', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Machine Learning Methods

Statistical methods work well for univariate data. For multivariate data, we need more sophisticated approaches.

### 4.1 Isolation Forest

**Intuition:** Anomalies are "few and different" — they are isolated by random splits much faster than normal points.

- Builds an ensemble of random trees (iTrees)
- Randomly selects a feature and a split value between min and max
- Anomalies require fewer splits to isolate → shorter path lengths
- Anomaly score $s(x) = 2^{-\frac{E(h(x))}{c(n)}}$ where $E(h(x))$ is the expected path length

**Key hyperparameter:** `contamination` — expected proportion of outliers.

### 4.2 One-Class SVM

**Intuition:** Learns a boundary around the normal data points (like a tight bubble) and flags anything outside.

- Maps data to a higher-dimensional feature space using a kernel (RBF by default)
- Finds the hyperplane that best separates the data from the origin
- `nu` parameter: upper bound on training errors, lower bound on support vectors
- Works best when the normal data is densely clustered

> **Note:** One-Class SVM is sensitive to outliers in the training set — ideal for **novelty detection** when training data is clean.

### 4.3 Elliptic Envelope

**Intuition:** Assumes the data follows a multivariate Gaussian distribution and fits an ellipse around it.

- Estimates the mean and covariance matrix (robustly using MCD — Minimum Covariance Determinant)
- Computes the Mahalanobis distance for each point:
  $$D_M(x) = \sqrt{(x - \mu)^T \Sigma^{-1} (x - \mu)}$$
- Points beyond a threshold are flagged as outliers
- Best when data is roughly Gaussian (unimodal)

### 4.4 Local Outlier Factor (LOF)

**Intuition:** Compares the local density of a point to its neighbors. Anomalies have much lower density than their neighbors.

- For each point, compute the **local reachability density** (lrd)
- LOF score = average (lrd of neighbors) / (lrd of the point)
- LOF ≈ 1 → similar density to neighbors (normal)
- LOF > 1 → lower density than neighbors (potential outlier)
- Captures local anomalies well (unlike global methods)

### 4.5 DBSCAN for Outlier Detection

**Intuition:** DBSCAN clusters dense regions and labels points in low-density regions as noise (outliers).

- `eps`: maximum distance between two points to be considered neighbors
- `min_samples`: minimum points to form a dense region
- Points not assigned to any cluster are labeled as outliers (-1)
- Naturally suited for outlier detection without a separate contamination parameter

---
## 5. Comparing All Methods on Synthetic Data

We create a 2D dataset with known outliers and visualize the decision boundary of each method.

In [ ]:
def generate_outlier_data(n_normal=300, n_outliers=20, random_state=42):
    rs = np.random.RandomState(random_state)
    X_normal, _ = make_blobs(n_samples=n_normal, centers=1, cluster_std=1.2,
                             random_state=random_state, center_box=(-2, 2))
    X_normal = X_normal @ np.array([[1.5, 0.8], [0.5, 1.2]])

    X_outliers = rs.uniform(low=-8, high=8, size=(n_outliers, 2))

    X = np.vstack([X_normal, X_outliers])
    y = np.hstack([np.zeros(n_normal), np.ones(n_outliers)])
    return X, y

X, y_true = generate_outlier_data(n_normal=300, n_outliers=30, random_state=42)

outlier_ratio = y_true.mean()
print(f"Dataset shape: {X.shape}")
print(f"Outlier ratio (contamination): {outlier_ratio:.3f}")
print(f"Normal points: {(y_true == 0).sum()}, Outliers: {(y_true == 1).sum()}")

In [ ]:
def plot_decision_boundary(ax, model, X, y_true, title, method_type='sklearn'):
    h = 0.1
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

    if method_type == 'sklearn':
        Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
        Z = Z == -1  # sklearn convention: -1 = outlier, 1 = inlier
    elif method_type == 'lof':
        Z = model.fit_predict(np.c_[xx.ravel(), yy.ravel()])
        Z = Z == -1
    elif method_type == 'dbscan':
        Z = model.fit_predict(np.c_[xx.ravel(), yy.ravel()])
        Z = Z == -1

    Z = Z.astype(int).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.2, cmap=ListedColormap(['#5DADE2', '#F1948A']))
    ax.contour(xx, yy, Z, colors='gray', linewidths=0.5, alpha=0.5)

    ax.scatter(X[y_true == 0, 0], X[y_true == 0, 1], c='#2E86C1', s=20,
               alpha=0.5, label='Normal (true)')
    ax.scatter(X[y_true == 1, 0], X[y_true == 1, 1], c='#E74C3C', s=40,
               alpha=0.8, marker='X', label='Outlier (true)')

    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    ax.legend(loc='upper right', fontsize=8)
    return ax


# Initialize all models
contamination = 0.1
models = {
    'Isolation Forest': IsolationForest(contamination=contamination, random_state=42),
    'One-Class SVM': OneClassSVM(nu=contamination, kernel='rbf', gamma='auto'),
    'Elliptic Envelope': EllipticEnvelope(contamination=contamination, random_state=42),
    'LOF': LocalOutlierFactor(contamination=contamination, novelty=False),
    'DBSCAN': DBSCAN(eps=1.2, min_samples=5)
}

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

types = ['sklearn', 'sklearn', 'sklearn', 'lof', 'dbscan']

for i, ((name, model), mtype) in enumerate(zip(models.items(), types)):
    if mtype == 'sklearn' or mtype == 'lof':
        if name == 'LOF':
            y_pred = model.fit_predict(X)
            y_pred_binary = (y_pred == -1).astype(int)
        else:
            model.fit(X)
            y_pred = model.predict(X)
            y_pred_binary = (y_pred == -1).astype(int)
    else:  # dbscan
        y_pred = model.fit_predict(X)
        y_pred_binary = (y_pred == -1).astype(int)

    accuracy = (y_pred_binary == y_true).mean()
    plot_decision_boundary(axes[i], model, X, y_true, f'{name}\n(Accuracy: {accuracy:.3f})', mtype)

# Remove extra subplot
axes[-1].set_visible(False)

plt.suptitle('Anomaly Detection Methods Comparison (Decision Boundaries)',
             fontsize=18, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### Observations from the Comparison

- **Isolation Forest** works well on this scattered dataset — it handles the irregular outlier distribution well.
- **One-Class SVM** draws a tight boundary but can be tuned with `gamma` and `nu`.
- **Elliptic Envelope** assumes an elliptical shape — struggles when outliers don't follow the Gaussian contour.
- **LOF** adapts locally — useful when density varies across the dataset.
- **DBSCAN** naturally labels low-density points as noise.

---
## 6. The Contamination Parameter

Most anomaly detection algorithms require setting the **contamination** parameter — the expected proportion of outliers in the data.

- **Set too low** → many true outliers will be missed (false negatives)
- **Set too high** → many normal points will be flagged (false positives)
- **Auto-estimation**: If unknown, use domain knowledge, or try a range and evaluate with business metrics
- Some methods (Isolation Forest, Elliptic Envelope) can also use `contamination='auto'` (estimates from the data)

In [ ]:
contamination_values = np.linspace(0.02, 0.3, 10)
results = []

for c in contamination_values:
    iso = IsolationForest(contamination=c, random_state=42)
    y_pred = iso.fit_predict(X)
    y_pred_binary = (y_pred == -1).astype(int)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred_binary, average='binary')
    results.append({'contamination': c, 'precision': prec, 'recall': rec, 'f1': f1})

df_cont = pd.DataFrame(results)
df_cont

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df_cont['contamination'], df_cont['precision'], 'o-', label='Precision', lw=2)
ax.plot(df_cont['contamination'], df_cont['recall'], 's-', label='Recall', lw=2)
ax.plot(df_cont['contamination'], df_cont['f1'], '^-', label='F1-Score', lw=3)
ax.axvline(x=outlier_ratio, color='gray', ls='--', alpha=0.7, label=f'True ratio: {outlier_ratio:.3f}')
ax.set_xlabel('Contamination Parameter')
ax.set_ylabel('Score')
ax.set_title('Effect of Contamination Parameter on Detection Performance')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

---
## 7. Evaluating Anomaly Detection

When ground truth labels are available, we evaluate using standard classification metrics:

In [ ]:
models_eval = {
    'Isolation Forest': IsolationForest(contamination=0.1, random_state=42),
    'One-Class SVM': OneClassSVM(nu=0.1, kernel='rbf', gamma='auto'),
    'Elliptic Envelope': EllipticEnvelope(contamination=0.1, random_state=42),
    'LOF': LocalOutlierFactor(contamination=0.1),
    'DBSCAN': DBSCAN(eps=1.2, min_samples=5)
}

evaluation_results = []

for name, model in models_eval.items():
    if name == 'LOF':
        y_pred = model.fit_predict(X)
        y_pred_binary = (y_pred == -1)
    elif name == 'DBSCAN':
        y_pred = model.fit_predict(X)
        y_pred_binary = (y_pred == -1)
    else:
        y_pred = model.fit_predict(X)
        y_pred_binary = (y_pred == -1)

    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred_binary, average='binary')
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred_binary).ravel()

    evaluation_results.append({
        'Model': name,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn
    })

df_eval = pd.DataFrame(evaluation_results)
df_eval

### ROC Curve for Anomaly Detection

Most anomaly detectors provide an **anomaly score** (decision function) that we can threshold. This allows us to plot the ROC curve.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

roc_models = {
    'Isolation Forest': IsolationForest(contamination=0.1, random_state=42),
    'One-Class SVM': OneClassSVM(nu=0.1, kernel='rbf', gamma='auto'),
    'Elliptic Envelope': EllipticEnvelope(contamination=0.1, random_state=42),
    'LOF': LocalOutlierFactor(contamination=0.1)
}

for name, model in roc_models.items():
    if name == 'LOF':
        y_score = model.fit_predict(X)
        # LOF negative_outlier_factor_ is available after fit_predict
        y_score = -model.negative_outlier_factor_
    else:
        model.fit(X)
        y_score = -model.decision_function(X)  # negate so higher = more anomalous

    fpr, tpr, _ = roc_curve(y_true, y_score)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, lw=2, label=f'{name} (AUC = {roc_auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves for Anomaly Detection Methods')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 8. Novelty Detection vs Outlier Detection

It's important to distinguish two related but different tasks:

| Aspect | Outlier Detection | Novelty Detection |
|--------|------------------|-------------------|
| **Training data** | Contains outliers | Clean (no outliers) |
| **Goal** | Identify outliers within the dataset | Detect novel points in new data |
| **Approach** | Fit the model, predict on same data | Fit on clean data, predict on new points |
| **Example** | Finding bad transactions in a database | Flagging a new never-before-seen attack pattern |
| **LOF** | `novelty=False` (default) | `novelty=True` (then use `.predict()` on new data) |

**Key takeaway:** For novelty detection, ensure your training set is clean. For outlier detection, the model must be robust to outliers in the training data.

In [ ]:
# Novelty detection example: train on clean data, test on new data
X_clean, _ = make_blobs(n_samples=200, centers=1, cluster_std=1.0, random_state=42)
X_novel = np.random.uniform(low=-5, high=5, size=(30, 2))
X_test = np.vstack([X_clean[:20], X_novel])
y_test = np.hstack([np.zeros(20), np.ones(30)])

# Train One-Class SVM on CLEAN data (novelty detection mode)
ocsvm = OneClassSVM(nu=0.1, kernel='rbf', gamma='scale')
ocsvm.fit(X_clean)
y_pred = ocsvm.predict(X_test)
y_pred_binary = (y_pred == -1).astype(int)

print("Novelty Detection Results:")
print(f"Accuracy: {(y_pred_binary == y_test).mean():.3f}")
print(classification_report(y_test, y_pred_binary, target_names=['Normal', 'Novel']))

---
## 9. Feature Engineering for Anomaly Detection

Good features make anomaly detection much more effective. Common strategies include:

1. **Aggregation features** — counts, sums, averages over a time window
2. **Rate features** — velocity, frequency of events per unit time
3. **Ratio features** — ratios between amounts (e.g., transaction amount / account balance)
4. **Statistical features** — rolling mean, rolling std, distance from moving average
5. **Encoding temporal patterns** — hour of day, day of week, seasonality
6. **Distance-based features** — distance to nearest cluster centroid (kNN)
7. **PCA / Autoencoder reconstruction error** — how well a point fits the learned manifold

In [ ]:
# Demonstrate: using distance-to-centroid as a feature for anomaly detection
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=1, random_state=42)
kmeans.fit(X)
distances = np.linalg.norm(X - kmeans.cluster_centers_[0], axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
scatter = ax.scatter(X[:, 0], X[:, 1], c=distances, cmap='viridis', s=30, edgecolors='black')
ax.scatter(kmeans.cluster_centers_[0, 0], kmeans.cluster_centers_[0, 1],
           marker='*', s=300, c='red', edgecolors='black', label='Centroid')
ax.set_title('Distance to Centroid as Feature')
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
plt.colorbar(scatter, ax=ax, label='Distance')
ax.legend()

ax = axes[1]
threshold = np.percentile(distances, 90)
anomalous = distances > threshold
ax.scatter(X[~anomalous, 0], X[~anomalous, 1], c='#2E86C1', s=20, alpha=0.5, label='Normal')
ax.scatter(X[anomalous, 0], X[anomalous, 1], c='#E74C3C', s=40, marker='X', label='Flagged')
ax.set_title(f'Top 10% Most Distant Points (Distance > {threshold:.2f})')
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.legend()

plt.suptitle('Feature Engineering: Distance-Based Anomaly Detection', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 10. Real-World Example 1: Side-by-Side Comparison with All Methods

In [ ]:
# Generate a more complex 2D dataset with outliers
np.random.seed(42)

# Two normal clusters
X1, _ = make_blobs(n_samples=200, centers=[(0, 0)], cluster_std=0.8, random_state=42)
X2, _ = make_blobs(n_samples=200, centers=[(4, 3)], cluster_std=0.6, random_state=42)
X_normal = np.vstack([X1, X2])

# Scattered outliers
X_outliers = np.array([
    [-2, -2], [5, -1], [-1, 5], [6, 6], [-3, 3], [7, 0], [0, -3], [3, 6],
    [-2.5, 0], [4, -2], [2, 5], [7, 2], [-1, -3], [5, 5], [-3, -1]
])

X_full = np.vstack([X_normal, X_outliers])
y_full = np.hstack([np.zeros(400), np.ones(15)])

print(f"Normal: {(y_full == 0).sum()}, Outliers: {(y_full == 1).sum()}")
print(f"True contamination: {y_full.mean():.4f}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

all_models = {
    'Isolation Forest': IsolationForest(contamination=0.05, random_state=42),
    'One-Class SVM': OneClassSVM(nu=0.05, kernel='rbf', gamma='scale'),
    'Elliptic Envelope': EllipticEnvelope(contamination=0.05, random_state=42),
    'LOF': LocalOutlierFactor(contamination=0.05),
    'DBSCAN': DBSCAN(eps=0.8, min_samples=5)
}

model_types = ['sklearn', 'sklearn', 'sklearn', 'lof', 'dbscan']

for i, ((name, model), mtype) in enumerate(zip(all_models.items(), model_types)):
    if mtype == 'lof':
        y_pred = model.fit_predict(X_full)
        y_pred_binary = (y_pred == -1).astype(int)
    elif mtype == 'dbscan':
        y_pred = model.fit_predict(X_full)
        y_pred_binary = (y_pred == -1).astype(int)
    else:
        model.fit(X_full)
        y_pred = model.predict(X_full)
        y_pred_binary = (y_pred == -1).astype(int)

    plot_decision_boundary(axes[i], model, X_full, y_full, name, mtype)

    prec, rec, f1, _ = precision_recall_fscore_support(y_full, y_pred_binary, average='binary')
    axes[i].text(0.05, 0.95, f'F1: {f1:.3f}\nPr: {prec:.3f}\nRe: {rec:.3f}',
                 transform=axes[i].transAxes, fontsize=10, verticalalignment='top',
                 bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

axes[-1].set_visible(False)
plt.suptitle('Real-World Example 1: Multi-Cluster Outlier Detection',
             fontsize=18, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## 11. Real-World Example 2: Fraud Detection Style

We simulate a tabular dataset with numeric features typical of a fraud detection scenario:
- Transaction amount, account age, number of transactions, location distance, etc.
- Inject synthetic fraud cases
- Compare all 5 detection methods

In [ ]:
np.random.seed(42)
n_samples = 1000

df_fraud = pd.DataFrame({
    'transaction_amount': np.random.exponential(100, n_samples),
    'account_age_days': np.random.randint(1, 3650, n_samples),
    'num_transactions_30d': np.random.poisson(20, n_samples),
    'avg_transaction_amount': np.random.exponential(80, n_samples),
    'location_distance_km': np.random.exponential(50, n_samples),
    'hour_of_day': np.random.randint(0, 24, n_samples),
    'is_weekend': np.random.binomial(1, 0.3, n_samples)
})

# Inject fraud: extreme values
n_fraud = 30
fraud_indices = np.random.choice(n_samples, n_fraud, replace=False)

df_fraud.loc[fraud_indices, 'transaction_amount'] = np.random.uniform(500, 2000, n_fraud)
df_fraud.loc[fraud_indices, 'num_transactions_30d'] = np.random.poisson(50, n_fraud)
df_fraud.loc[fraud_indices, 'location_distance_km'] = np.random.uniform(200, 1000, n_fraud)
df_fraud['is_fraud'] = 0
df_fraud.loc[fraud_indices, 'is_fraud'] = 1

print(f"Fraud rate: {df_fraud['is_fraud'].mean():.4f}")
df_fraud.head(10)

In [ ]:
# Scale features and fit models
feature_cols = ['transaction_amount', 'account_age_days', 'num_transactions_30d',
                'avg_transaction_amount', 'location_distance_km', 'hour_of_day', 'is_weekend']
scaler = StandardScaler()
X_fraud = scaler.fit_transform(df_fraud[feature_cols])
y_fraud = df_fraud['is_fraud'].values

fraud_models = {
    'Isolation Forest': IsolationForest(contamination=0.03, random_state=42),
    'One-Class SVM': OneClassSVM(nu=0.03, kernel='rbf', gamma='scale'),
    'Elliptic Envelope': EllipticEnvelope(contamination=0.03, random_state=42),
    'LOF': LocalOutlierFactor(contamination=0.03),
    'DBSCAN': DBSCAN(eps=2.5, min_samples=10)
}

fraud_results = []
for name, model in fraud_models.items():
    if name == 'LOF':
        y_pred = model.fit_predict(X_fraud)
        y_pred_binary = (y_pred == -1).astype(int)
    elif name == 'DBSCAN':
        y_pred = model.fit_predict(X_fraud)
        y_pred_binary = (y_pred == -1).astype(int)
        n_clusters = len(set(y_pred)) - (1 if -1 in y_pred else 0)
    else:
        y_pred = model.fit_predict(X_fraud)
        y_pred_binary = (y_pred == -1).astype(int)

    cm = confusion_matrix(y_fraud, y_pred_binary)
    prec, rec, f1, _ = precision_recall_fscore_support(y_fraud, y_pred_binary, average='binary')
    tn, fp, fn, tp = cm.ravel()

    fraud_results.append({
        'Model': name,
        'Precision': round(prec, 3),
        'Recall': round(rec, 3),
        'F1': round(f1, 3),
        'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn
    })

df_fraud_results = pd.DataFrame(fraud_results)
df_fraud_results

In [ ]:
# Visualize feature distributions for fraud vs normal
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    ax = axes[i]
    for label, color, name in [(0, '#2E86C1', 'Normal'), (1, '#E74C3C', 'Fraud')]:
        subset = df_fraud[df_fraud['is_fraud'] == label][col]
        ax.hist(subset, bins=30, alpha=0.6, color=color, label=name, density=True)
    ax.set_title(col.replace('_', ' ').title())
    ax.set_xlabel(col)
    ax.legend(fontsize=8)

axes[-1].set_visible(False)
plt.suptitle('Fraud Detection: Feature Distributions (Normal vs Fraud)',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Visual breakdown of detected outliers by model
fig, ax = plt.subplots(figsize=(12, 5))

x = np.arange(len(fraud_results))
width = 0.25

ax.bar(x - width, [r['Precision'] for r in fraud_results], width, label='Precision', color='#2E86C1')
ax.bar(x, [r['Recall'] for r in fraud_results], width, label='Recall', color='#F39C12')
ax.bar(x + width, [r['F1'] for r in fraud_results], width, label='F1', color='#27AE60')

ax.set_xticks(x)
ax.set_xticklabels([r['Model'] for r in fraud_results], rotation=15, ha='right')
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Fraud Detection: Performance Comparison')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

for i, r in enumerate(fraud_results):
    ax.annotate(f"TP:{r['TP']} FP:{r['FP']}\nFN:{r['FN']} TN:{r['TN']}",
                xy=(i, r['F1'] + 0.05), ha='center', fontsize=8,
                bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.7))

plt.tight_layout()
plt.show()

---
## Summary: Choosing the Right Method

| Method | Best For | Assumptions | Pros | Cons |
|--------|----------|-------------|------|------|
| **Z-score / IQR** | Quick univariate checks | Gaussian / any distribution | Simple, interpretable | Univariate only |
| **Isolation Forest** | High-dimensional data, mixed types | Anomalies are few and distinct | Fast, scales well, little tuning | Randomness can cause variance |
| **One-Class SVM** | Clean training sets, novelty detection | Normal data is clustered | Flexible boundary (kernel trick) | Sensitive to outliers in training, slow on large data |
| **Elliptic Envelope** | Gaussian-like distributions | Multivariate normality | Fast, interpretable | Poor for non-elliptical shapes |
| **LOF** | Local anomalies, varying density | Local density differences | Captures local outliers | Sensitive to `n_neighbors`, expensive on large data |
| **DBSCAN** | Clustered normal data with scattered noise | Dense regions separated by sparse areas | No contamination param needed | Struggles with varying density clusters |

---
## Practice Exercises

1. **Z-score tuning**: On the `df_stats` dataset, experiment with different Z-score thresholds (2.0, 2.5, 3.0, 3.5). How many outliers does each detect? What happens to false positives as you lower the threshold?

2. **Contamination grid search**: For the fraud detection dataset (`df_fraud`), run Isolation Forest with contamination values from 0.01 to 0.15. Find the value that maximizes F1-score. Plot the precision-recall trade-off.

3. **DBSCAN parameter tuning**: On the 2D synthetic dataset (`X_full`, `y_full`), vary `eps` from 0.3 to 2.0 and `min_samples` from 3 to 15. Find the best combination using F1-score. How does the number of clusters change?

4. **LOF vs Isolation Forest on local anomalies**: Create a dataset where one region is very dense and another is sparse. Inject an anomaly in the sparse region. Which method detects it better? (Hint: LOF excels at local anomalies)

5. **Feature engineering challenge**: Add two new features to the fraud dataset: (a) `amount_to_avg_ratio = transaction_amount / avg_transaction_amount`, and (b) `transaction_frequency = num_transactions_30d / account_age_days * 30`. Re-run Isolation Forest. Does F1 improve? Which features are most important according to the model's feature importances?